In [ ]:
import jax
import os

In [ ]:
import sys
sys.path.append("../")

%load_ext autoreload
%autoreload 2

import pulse_simulator as ps

In [ ]:
from datetime import datetime
import functools
import itertools
import logging
import numpy as np
import pickle as pkl

import qiskit as qk
from qiskit import pulse
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d

import qutip as qt
import re 
from scipy.integrate import simpson
from scipy.linalg import sqrtm, expm, logm
from tqdm import tqdm
import warnings

import matplotlib.pyplot as plt

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()
dt = backend.configuration().dt * ns
desired_1qb_duration = 225  # dt
desired_2qb_duration = 225 * 4  # dt

# Gate time in SI units (granularity restriction)
duration_1qb = ps.qiskit_closest_multiple_of_16(desired_1qb_duration * dt / ns, backend)
duration_2qb = ps.qiskit_closest_multiple_of_16(desired_2qb_duration * dt / ns, backend)

# Shapes

In [ ]:
names_1qb = ["X1", "X2", "SX1", "SX2", "I1", "I2"]
angles_1qb = [1.5 * np.pi, 0.5 * np.pi, 1.25 * np.pi, 0.25 * np.pi, 0.2, 0.0]

cal_1qb_default = ps.default_single_qubit_gateset(
    names_1qb,
    duration_1qb,
    dt,
)

In [ ]:
# Load shapes
def fname(k):
    return f"../pico-pulses/saved-pulses-2024-01-16/a_two_qubit_{k}.csv"

base_2qb = {k: ps.from_csv(fname(k))[0] for k in ["default", "robust"]}
default_2qb = 2 * [base_2qb["default"]]
names_2qb = ["+", "-"]

cal_2qb_default = ps.calibrate_gateset(
    gateset=default_2qb,
    names=names_2qb,
    angles=[0.125 * np.pi, -0.125 * np.pi],
    duration=duration_2qb,
    dt=dt,
    samples=32,
)

# Simulate schedules

In [ ]:
registers = [0]

In [ ]:
solver_r0 = ps.build_solver(
    registers,
    backend,
    include_crosstalk=False,
    include_dissipation=False,
    simulation_dt=dt,
    verbose=True
)

In [ ]:
test_sched = ps.build_pulse_schedule(
    [{0: np.pi / 2}],
    qubit_gates=cal_1qb_default,
    cr_gates=cal_2qb_default,
    registers=registers,
    backend=backend,
    name="test",
    dt=dt,
    insert_identities=True,
    verbose=True
)

In [ ]:
y0 = ps.qiskit_identity_operator(len(registers))

solver_r0.model.evaluation_mode = "dense"
res_0 = solver_r0.solve(
    t_span=[0.0, test_sched.duration * dt],
    t_eval=[test_sched.duration * dt],
    y0=y0,
    signals=test_sched,
    atol=1e-9,
    rtol=1e-9,
    method="jax_odeint",
)

In [ ]:
θ = (1j * logm(res_0.y[-1].data)).real
θ

In [ ]:
np.mod(θ[0,0] / np.pi, 2)


In [ ]:
# Check constant envelope functionality
DURATION_PHASE_GATE = 11

test = ps.constant_envelope(
    DURATION_PHASE_GATE,
    angle=np.pi,
    name="phase",
    dt=dt,
)

np.trapz(ps.get_samples(test).real, dx=dt) / np.pi


In [ ]:
DURATION_PHASE_GATE = 11

test = ps.constant_envelope(
    DURATION_PHASE_GATE,
    angle=np.pi,
    name="phase",
    dt=1,
)

np.trapz(ps.get_samples(test).real, dx=dt) / np.pi
